# 04 · Variational Autoencoders for Anomaly Detection

## What this notebook covers
VAEs extend autoencoders by learning a probabilistic latent space. Instead of encoding a point to a fixed vector, the encoder outputs a distribution (mean and variance). This gives a richer anomaly score and better calibrated uncertainty.

## The ELBO anomaly score
The VAE training objective is the ELBO (Evidence Lower BOund):

    ELBO = E[log p(x|z)] - KL(q(z|x) || p(z))
         = Reconstruction term - Regularisation term

Both terms contribute to anomaly scoring:
- High reconstruction error → the decoder can't reproduce the input
- High KL divergence → the encoder posterior is far from the prior (the point doesn't fit the learned latent structure)

The combined ELBO score is more informative than reconstruction error alone. Anomalies typically show both high reconstruction loss AND high KL divergence.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
torch.manual_seed(42); np.random.seed(42)

X_normal, _ = make_blobs(n_samples=1000, centers=2, cluster_std=0.5, random_state=0, n_features=16)
X_anom = np.random.uniform(-6, 6, (50, 16))
X_all = np.vstack([X_normal, X_anom])
y_all = np.array([0]*1000 + [1]*50)

scaler = StandardScaler()
X_train = torch.FloatTensor(scaler.fit_transform(X_normal))
X_test  = torch.FloatTensor(scaler.transform(X_all))

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim=16, latent_dim=4):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.fc_mu  = nn.Linear(32, latent_dim)
        self.fc_var = nn.Linear(32, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32), nn.ReLU(),
            nn.Linear(32, input_dim)
        )
        self.latent_dim = latent_dim

    def encode(self, x):
        h = F.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_var(h)

    def reparameterise(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterise(mu, log_var)
        return self.decoder(z), mu, log_var

    def elbo_score(self, x):
        """Per-sample negative ELBO as anomaly score (higher = more anomalous)."""
        with torch.no_grad():
            recon, mu, log_var = self.forward(x)
            recon_loss = ((x - recon) ** 2).mean(dim=1)
            kl_loss = -0.5 * (1 + log_var - mu.pow(2) - log_var.exp()).sum(dim=1)
            return (recon_loss + kl_loss).numpy(), recon_loss.numpy(), kl_loss.numpy()

vae = VAE(input_dim=16, latent_dim=4)
optimiser = torch.optim.Adam(vae.parameters(), lr=1e-3)
loader = DataLoader(TensorDataset(X_train), batch_size=64, shuffle=True)

losses = []
for epoch in range(100):
    epoch_loss = 0
    for (batch,) in loader:
        recon, mu, log_var = vae(batch)
        recon_l = F.mse_loss(recon, batch, reduction="sum")
        kl_l    = -0.5 * (1 + log_var - mu.pow(2) - log_var.exp()).sum()
        loss = (recon_l + kl_l) / batch.size(0)
        optimiser.zero_grad(); loss.backward(); optimiser.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(loader))
print(f"Final ELBO loss: {losses[-1]:.4f}")

In [ ]:
elbo_scores, recon_scores, kl_scores = vae.elbo_score(X_test)

auc_elbo  = roc_auc_score(y_all, elbo_scores)
auc_recon = roc_auc_score(y_all, recon_scores)
auc_kl    = roc_auc_score(y_all, kl_scores)
print(f"VAE ELBO score AUC   : {auc_elbo:.4f}")
print(f"Reconstruction only  : {auc_recon:.4f}")
print(f"KL divergence only   : {auc_kl:.4f}")
print("ELBO combines both signals — typically best")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].hist(elbo_scores[y_all==0], bins=40, alpha=0.7, color="steelblue", label="Normal", density=True)
axes[0].hist(elbo_scores[y_all==1], bins=20, alpha=0.7, color="tomato",    label="Anomaly", density=True)
axes[0].set_title(f"ELBO score distribution (AUC={auc_elbo:.3f})")
axes[0].legend(fontsize=8)

# Latent space visualisation (first 2 dims of mu)
with torch.no_grad():
    mu_all, _ = vae.encode(X_test)
mu_np = mu_all.numpy()
axes[1].scatter(mu_np[y_all==0, 0], mu_np[y_all==0, 1], c="steelblue", s=10, alpha=0.5, label="Normal")
axes[1].scatter(mu_np[y_all==1, 0], mu_np[y_all==1, 1], c="red", s=50, marker="x", label="Anomaly")
axes[1].set_title("Latent space (μ, first 2 dims)"); axes[1].legend(fontsize=8)

# Score decomposition scatter
axes[2].scatter(recon_scores[y_all==0], kl_scores[y_all==0], c="steelblue", s=10, alpha=0.4, label="Normal")
axes[2].scatter(recon_scores[y_all==1], kl_scores[y_all==1], c="red", s=50, marker="x", label="Anomaly")
axes[2].set_xlabel("Reconstruction loss"); axes[2].set_ylabel("KL divergence")
axes[2].set_title("Score decomposition"); axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{base}/04_vae.png", dpi=120)
plt.show()